# 🤖 K-Fold Training for Stance Classification with BERT
This notebook loads the enriched dataset, tokenizes it, and performs
5-fold cross-validation using Hugging Face Transformers' Trainer API.
We log to TensorBoard and Weights & Biases, and save models + predictions.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, DataCollatorWithPadding)
import torch
import evaluate
import wandb
from datasets import Dataset

wandb.init(project='stance-kfold-bert', reinit=True)

# Use GPU via MPS on Mac M1
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# Load enriched dataset
df = pd.read_csv("cleaned_stance_dataset_enriched.csv")
label2id = {0: "Negative", 1: "Positive", 2: "Neutral"}

# Tokenizer & Model
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples['content'], truncation=True, padding=True, max_length=128)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Create Dataset object
dataset = Dataset.from_pandas(df[['content', 'label']])
dataset = dataset.map(preprocess_function, batched=True)

/Users/sergioparigi/Desktop/TESI/Tesi Sergio Final/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: Paste an API key from your profile and hit enter:wandb: P

In [ ]:
# K-Fold Training
kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"\n🔁 Fold {fold+1}")
    train_dataset = dataset.select(train_idx)
    eval_dataset = dataset.select(val_idx)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint, num_labels=3
    ).to(device)

    args = TrainingArguments(
        output_dir=f"output/fold{fold+1}",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        load_best_model_at_end=True,
        report_to=["wandb", "tensorboard"],
        logging_dir=f"logs/fold{fold+1}"
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds, average='weighted')
        }

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_result = trainer.evaluate()
    results.append(eval_result)

    # Save predictions
    preds = trainer.predict(eval_dataset)
    pred_labels = np.argmax(preds.predictions, axis=-1)
    df_pred = pd.DataFrame({
        "true_label": preds.label_ids,
        "predicted_label": pred_labels
    })
    df_pred.to_csv(f"output/fold{fold+1}/predictions.csv", index=False)

    # Save model
    trainer.save_model(f"output/fold{fold+1}/model")

# Save all fold results
pd.DataFrame(results).to_csv("output/kfold_results.csv", index=False)